# 📝 Cypher 심화 과제 LV2 정답: 물류 배송망 (강사용)

각 문제의 **모범답안 + 해설**입니다. 도메인은 **물류 배송망**입니다.

- 노드: `Warehouse`(창고)·`Hub`(허브)·`City`(도시). 관계: `(출발)-[:ROUTE {time}]->(도착)`(방향 있음, `time` 은 분).
- 채점은 `run_cypher` 결과를 파이썬으로 결정적 비교합니다.

아래 세 셀(연결 → 초기화 → 시드 적재)을 먼저 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 앞 단원에서 만든 그래프(day28 의 Movies 예제, day29 의 과제 결과)도 함께 사라집니다. 되돌릴 수 없으니 `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요.

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙은 관계까지 함께 지우라는 뜻입니다.
run_cypher("MATCH (n) DETACH DELETE n")
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

오늘 풀 문제의 그래프입니다. 창고 2곳·허브 3곳·도시 5곳이 육상 노선 9개와 항공 노선 1개로 이어져 있습니다.

<img src="../images/배송망_그래프.png" width="820">

In [ ]:
# [제공 코드] 물류 배송망 시드 적재: 이 셀은 실행만 하세요(그래프를 처음부터 만듭니다).
run_cypher("""
CREATE (icn:Warehouse {name:'인천창고'}),
       (busan:Warehouse {name:'부산창고'})
CREATE (seoul:Hub {name:'서울허브'}),
       (daejeon:Hub {name:'대전허브'}),
       (gwangju:Hub {name:'광주허브'})
CREATE (suwon:City {name:'수원시'}),
       (cheonan:City {name:'천안시'}),
       (jeonju:City {name:'전주시'}),
       (mokpo:City {name:'목포시'}),
       (jeju:City {name:'제주시'})
CREATE (icn)-[:ROUTE {time:40}]->(seoul),
       (icn)-[:ROUTE {time:90}]->(daejeon),
       (busan)-[:ROUTE {time:70}]->(gwangju),
       (seoul)-[:ROUTE {time:30}]->(suwon),
       (seoul)-[:ROUTE {time:80}]->(cheonan),
       (daejeon)-[:ROUTE {time:50}]->(cheonan),
       (daejeon)-[:ROUTE {time:70}]->(jeonju),
       (gwangju)-[:ROUTE {time:40}]->(jeonju),
       (gwangju)-[:ROUTE {time:55}]->(mokpo)
CREATE (icn)-[:AIR_ROUTE {time:25}]->(mokpo)
""")
print("배송망 적재 완료. 노드:", len(run_cypher("MATCH (n) RETURN n")),
      "개, 육상 노선:", len(run_cypher("MATCH ()-[r:ROUTE]->() RETURN r")),
      "개, 항공 노선:", len(run_cypher("MATCH ()-[r:AIR_ROUTE]->() RETURN r")), "개")


## 데이터 살펴보기
아래 셀은 **실행만** 하세요. 노선(출발 → 도착, 소요 시간)을 훑어봅니다. 노선은 두 종류입니다: 육상 노선 `ROUTE` 와 항공 노선 `AIR_ROUTE`.

In [ ]:
# [제공 코드] 노선을 먼저 훑어봅니다(실행만 하세요)
# 세로선(|)으로 두 관계 종류를 함께 열고, type(x) 로 그 행이 어느 노선인지 되짚는다
for r in run_cypher("MATCH (a)-[x:ROUTE|AIR_ROUTE]->(b) "
                    "RETURN type(x) AS 종류, a.name AS 출발, b.name AS 도착, x.time AS 시간 "
                    "ORDER BY 종류, 출발, 도착"):
    print(r['종류'], ':', r['출발'], '→', r['도착'], f"({r['시간']}분)")

---
# 1. 경로 탐색

가변길이·최단 경로·경로 위 조건을 배송망에 적용합니다(교안_01).

## 1-1. 창고에서 닿는 도시: 가변길이 + 라벨 조건
**배경**: `인천창고` 에서 노선을 여러 번 갈아타 **최종적으로 닿을 수 있는 도시**를 알고 싶습니다.

**요구사항**:
- `인천창고` 에서 `ROUTE` 관계를 **가변길이로 앞으로**(`*1..`) 따라가되, 도착 노드가 **`City`** 라벨인 것만 찾으세요(몇 홉이 걸릴지 모르니 길이를 열어 둡니다).
- 결과를 변수 **`rows1_1`** 에 담으세요. 도시 이름을 `DISTINCT` 로, 반환 컬럼 별칭은 **`name`** 으로.

**예시**: 닿을 수 있는 도시는 **3곳** 입니다. 허브는 `City` 라벨이 아니라서 결과에 나오지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 가변길이 ROUTE 를 따라가되 끝 노드 라벨을 City 로 못박고 DISTINCT 한다.

세부구현:
1. 인천창고를 이름으로 특정하고, ROUTE 를 가변길이(*1.., 단계 제한 없음) 앞 방향으로 따라간다.
2. 도착 노드 라벨을 City 로 못 박아 도시만 남긴다.
3. 도시 이름을 DISTINCT 로 반환한다(별칭 name). 결과를 rows1_1 에 담는다.
```

</details>

In [ ]:
# 가변길이로 몇 홉이든 따라가되 끝 노드를 City 라벨로 못 박는다
rows1_1 = run_cypher("""
MATCH (a:Warehouse {name:'인천창고'})-[:ROUTE*1..]->(c:City)
RETURN DISTINCT c.name AS name
""")
print(sorted(r['name'] for r in rows1_1))

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows1_1) == ['수원시', '전주시', '천안시'], \
    '가변길이(*1..)로 몇 홉이든 따라갔는지, 끝 노드에 City 라벨을 붙였는지, DISTINCT 와 별칭 name 을 줬는지 확인하세요'
print('✅ 통과!')

**해설**: 가변길이(`*1..`)로 몇 홉이든 따라가고, 끝 노드 라벨을 `City` 로 못 박아 도시만 남깁니다. 육상 노선(`ROUTE`)만 따라갔으므로 목포시는 여기서 안 나옵니다. 목포시로 들어오는 육상 노선은 광주허브발 하나뿐이라 인천창고 쪽에서는 닿지 못합니다(항공 노선까지 열면 답이 달라집니다. 그 이야기는 1-7 에서 합니다).

## 1-2. 최단 경로가 지나는 곳: 경로 노드와 구간 시간
**배경**: `인천창고` 에서 `전주시` 까지 **가장 빠르게(적은 홉)** 가는 경로가 **어디를 지나는지**, 그리고 **구간마다 몇 분 걸리는지**를 함께 봅니다.

**요구사항**:
- `shortestPath` 로 `인천창고` → `전주시` 경로를 구해 경로 변수 `p` 에 담으세요. 몇 홉이 걸릴지 모르니 관계는 **가변길이**(`-[:ROUTE*]->`)로 열고, `ROUTE` 는 방향 있는 노선이니 화살표를 지킵니다.
- 그 경로에서 **지나는 노드 이름**을 순서대로, 그리고 **구간별 `time`** 을 순서대로 함께 꺼내세요. 노드는 `nodes(p)` 에서, 구간 시간은 **`relationships(p)`** 에서 나옵니다.
- 여기에 더해, 구간 시간 중 **70분을 넘는 것만** 따로 뽑아 별칭 **`slow`** 로 받으세요. 리스트 표현식의 세로선 **앞에 `WHERE`** 를 넣으면 조건을 통과한 원소만 남습니다(`[r IN relationships(p) WHERE 조건 | 표현식]`).
- 결과를 변수 **`rows1_2`** 에 담으세요. 노드 이름 목록을 반환 컬럼 별칭 **`names`**, 구간 시간 목록을 **`times`**, 느린 구간 목록을 **`slow`** 로 받으면 `rows1_2[0]['names']` 처럼 꺼내 볼 수 있습니다.

**예시**: 노드 **3곳**과 구간 **2개**가 순서대로 나오고(구간 수는 노드 수보다 하나 적습니다), 그중 70분을 넘는 구간은 **1개**만 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- shortestPath 로 방향 있는 경로 p 를 잡고, 노드 목록과 관계 목록에서 각각 이름과 시간을 뽑는다.

세부구현:
1. 인천창고와 전주시를 이름으로 특정하고, ROUTE 를 방향(앞으로) 지켜 가변길이로 이은 경로를 shortestPath 로 감싸 p 에 담는다.
2. nodes(p) 로 지나는 노드를 얻고, 리스트 표현으로 이름만 뽑아 별칭 names 로 반환한다.
3. relationships(p) 로 지나는 관계를 얻고, 같은 방식으로 time 속성만 뽑아 별칭 times 로 반환한다.
4. 같은 관계 목록을 한 번 더 훑되, 세로선 앞에 WHERE 를 넣어 70분을 넘는 것만 남긴다.
   4-1. 세로선 뒤에서 꺼내는 값은 times 와 같은 time 이다. 관계가 아니라 시간 목록이 나와야 한다(별칭 slow).
5. 결과를 rows1_2 에 담는다.
```

</details>

In [ ]:
# ROUTE 는 방향 있는 노선이라 화살표를 지켜(-[:ROUTE*]->) 최단 경로를 찾는다
# 노드에서는 이름을, 관계에서는 time 을 각각 순서대로 뽑는다
# 마지막 줄은 같은 관계 목록에 조건을 걸어 훑는다. 세로선 앞의 WHERE 가 원소를 걸러 목록이 짧아진다
rows1_2 = run_cypher("""
MATCH p = shortestPath( (a:Warehouse {name:'인천창고'})-[:ROUTE*]->(b:City {name:'전주시'}) )
RETURN [n IN nodes(p) | n.name] AS names,
       [r IN relationships(p) | r.time] AS times,
       [r IN relationships(p) WHERE r.time > 70 | r.time] AS slow
""")
print(rows1_2[0]['names'], rows1_2[0]['times'], rows1_2[0]['slow'])

In [ ]:
# [자가채점]
assert rows1_2[0]['names'] == ['인천창고', '대전허브', '전주시'], \
    'shortestPath 로 감쌌는지, nodes(p) 에서 이름만 뽑아 별칭 names 로 줬는지, 화살표 방향을 지켰는지 확인하세요'
assert rows1_2[0]['times'] == [90, 70], \
    'relationships(p) 에서 각 관계의 time 을 순서대로 뽑아 별칭 times 로 줬는지 확인하세요'
assert rows1_2[0]['slow'] == [90], \
    '세로선 앞에 WHERE 를 넣어 70분을 넘는 구간만 남겼는지, 별칭이 slow 인지 확인하세요'
print('✅ 통과!')

**해설**: `ROUTE` 는 방향이 있으니 `-[:ROUTE*]->` 로 방향을 지켜 최단 경로를 찾습니다. 이 경로는 유일해 순서가 항상 같습니다.

`nodes(p)` 와 `relationships(p)` 는 **같은 경로를 노드로 보느냐 관계로 보느냐**의 차이입니다. 그래서 구간 수(2)는 노드 수(3)보다 항상 하나 적고, 관계에 붙은 속성(`time`)은 `relationships(p)` 로만 꺼낼 수 있습니다.

**흔한 실수**: 구간 시간을 `nodes(p)` 에서 꺼내려다 막힙니다. `time` 은 노드가 아니라 **관계**의 속성입니다.

`slow` 는 같은 관계 목록을 한 번 더 훑되 **세로선 앞에 `WHERE`** 를 넣은 것입니다. 조건을 통과한 원소만 남으므로 길이가 2에서 1로 줄었습니다. `WHERE` 를 세로선 **뒤**에 적으면 문법 오류입니다. 순서는 `[변수 IN 목록 WHERE 조건 | 표현식]` 하나뿐입니다.

## 1-3. 모든 구간이 70분 이하인 경로: `all`
**배경**: 냉장 화물은 **한 구간이라도** 70분을 넘으면 안 됩니다. 총 시간이 아니라 **구간마다** 지켜야 하는 조건입니다.

**요구사항**:
- **`(w:Warehouse)`** 에서 **`(c:City)`** 까지 **1~2단계**(`*1..2`) 경로를 열고, 경로 변수 `p` 에 담으세요. 라벨을 붙이지 않으면 허브에서 출발하는 경로까지 섞여 답이 달라집니다.
- `WHERE` 에 **`all(x IN relationships(p) WHERE x.time <= 70)`** 을 걸어 **모든 구간**이 70분 이하인 경로만 남기세요.
- 결과를 변수 **`rows1_3`** 에 담으세요. 출발 창고 이름을 **`w`**, 도착 도시 이름을 **`c`** 별칭으로 반환하세요.

**예시**: **3행**이 나옵니다. `all` 을 `any` 로 바꾸면 6행으로 늘어납니다(한 구간만 맞아도 통과하니까요).

<details><summary>힌트</summary>

```text
접근방법:
- 경로를 변수 p 에 담아야 relationships(p) 로 구간 목록을 꺼낼 수 있다.

세부구현:
1. MATCH p = (w:Warehouse)-[:ROUTE*1..2]->(c:City) 로 경로를 연다.
2. WHERE all(x IN relationships(p) WHERE x.time <= 70) 을 건다.
3. w.name 을 w, c.name 을 c 로 반환한다. 결과를 rows1_3 에 담는다.
```

</details>

In [ ]:
# 구간마다 지켜야 하는 조건은 relationships(p) 목록을 all 로 훑어 검사한다
# 경로를 p 에 담지 않으면 relationships(p) 를 꺼낼 수 없다
rows1_3 = run_cypher("""
MATCH p = (w:Warehouse)-[:ROUTE*1..2]->(c:City)
WHERE all(x IN relationships(p) WHERE x.time <= 70)
RETURN w.name AS w, c.name AS c
""")
print(sorted((r['w'], r['c']) for r in rows1_3))

In [ ]:
# [자가채점]
assert sorted((r['w'], r['c']) for r in rows1_3) == [('부산창고', '목포시'), ('부산창고', '전주시'), ('인천창고', '수원시')], \
    '경로를 p 에 담았는지, all 로 relationships(p) 를 훑었는지, 별칭이 w·c 인지 확인하세요 (any 로 쓰면 한 구간만 맞아도 통과해 답이 늘어납니다)'
print('✅ 통과!')

**해설**: `all` 은 목록의 **모든** 원소가 조건을 만족해야 참입니다. `any` 로 바꾸면 "한 구간이라도 70분 이하"가 되어 뜻이 완전히 달라집니다. "구간마다 지켜야 하는 조건"인지 "하나라도 있으면 되는 조건"인지를 먼저 정하고 함수를 고르세요.

## 1-4. 같은 길이의 최단 경로 전부: `allShortestPaths`
**배경**: `인천창고 → 천안시` 는 **서울허브를 거치는 길**과 **대전허브를 거치는 길**이 둘 다 **2단계**입니다. 어느 쪽으로 보낼지 정하려면 **둘 다** 봐야 합니다.

**요구사항**:
- 같은 짝을 **두 번** 구해 갈래 수를 비교하세요. `shortestPath` 결과는 변수 **`rows1_4_one`** 에, `allShortestPaths` 결과는 변수 **`rows1_4`** 에 담으세요.
- 두 조회 모두 지나는 노드 이름 목록을 반환 컬럼 별칭 **`names`** 로 받으세요.
- 화물은 화살표 방향으로만 가므로 **`-[:ROUTE*]->`** 로 방향을 지키세요.

**예시**: `shortestPath` 는 **1갈래**, `allShortestPaths` 는 **2갈래**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 함수는 이름만 다르고 쓰는 자리와 모양이 같다.

세부구현:
1. MATCH p = shortestPath( (a:Warehouse {name:'인천창고'})-[:ROUTE*]->(b:City {name:'천안시'}) ) 로 감싸고
   [n IN nodes(p) | n.name] 을 별칭 names 로 반환해 rows1_4_one 에 담는다.
2. 함수 이름만 allShortestPaths 로 바꿔 한 번 더 실행하고 결과를 rows1_4 에 담는다.
3. 두 변수의 len 을 나란히 찍어 갈래 수를 비교한다.
   돌아오는 순서는 정해져 있지 않으므로 경로를 출력할 때는 파이썬에서 정렬한다.
```

</details>

In [ ]:
# 같은 길이의 최단 경로가 여럿이면 shortestPath 는 그중 하나만 준다
rows1_4_one = run_cypher("""
MATCH p = shortestPath( (a:Warehouse {name:'인천창고'})-[:ROUTE*]->(b:City {name:'천안시'}) )
RETURN [n IN nodes(p) | n.name] AS names
""")
# 전부 받으려면 함수 이름만 allShortestPaths 로 바꾼다. 나머지 모양은 글자 하나 안 바뀐다
rows1_4 = run_cypher("""
MATCH p = allShortestPaths( (a:Warehouse {name:'인천창고'})-[:ROUTE*]->(b:City {name:'천안시'}) )
RETURN [n IN nodes(p) | n.name] AS names
""")
print('shortestPath:', len(rows1_4_one), '갈래 / allShortestPaths:', len(rows1_4), '갈래')
# 돌아오는 순서는 정해져 있지 않으므로 정렬해 찍는다
for names in sorted(r['names'] for r in rows1_4):
    print(' → '.join(names))

In [ ]:
# [자가채점]
assert len(rows1_4_one) == 1 and rows1_4_one[0]['names'] in [['인천창고', '대전허브', '천안시'], ['인천창고', '서울허브', '천안시']], \
    'rows1_4_one 은 shortestPath 로 구한 한 갈래여야 합니다. 별칭이 names 인지도 확인하세요'
assert sorted(r['names'] for r in rows1_4) == [['인천창고', '대전허브', '천안시'], ['인천창고', '서울허브', '천안시']], \
    'allShortestPaths 로 감쌌는지, 화살표 방향(->)을 지켰는지, 별칭이 names 인지 확인하세요'
print('✅ 통과!')

**해설**: `shortestPath` 는 최단 경로를 **한 갈래만** 돌려주고 어느 쪽을 줄지 우리가 정하지 못합니다. "경유지를 고를 수 있게 전부 보여 달라"는 요구에는 `allShortestPaths` 를 씁니다. 두 갈래 중 무엇을 고를지는 **총 소요 시간처럼 우리가 정한 기준**이 있어야 정해집니다(3-3 에서 그 기준을 만들어 봅니다).

**두 조회의 차이는 함수 이름 하나뿐입니다.** 그래서 `rows1_4_one` 은 1행, `rows1_4` 는 2행입니다. `shortestPath` 가 둘 중 어느 쪽을 돌려줄지는 정해져 있지 않으므로, 채점도 "둘 중 하나" 로만 봅니다.

## 1-5. 배송 가능 여부 판별
**배경**: `인천창고` 에서 `목포시` 로 화물을 보낼 수 있는지 판단해야 합니다.

**요구사항**:
- `인천창고` 와 `목포시` 를 잡고, 둘 사이 `ROUTE` 경로를 `OPTIONAL MATCH` + `shortestPath` 로 구한 뒤 **경로가 존재하는지**(`p IS NOT NULL`) 를 참/거짓으로 얻으세요.
- ⚠️ `ROUTE` 는 **방향 있는 노선**이라는 점을 잊지 마세요. 화물은 노선이 난 방향으로만 갈 수 있습니다. 방향을 어떻게 적느냐로 답이 달라지는 문제입니다.
- 결과를 변수 **`rows1_5`** 에 담으세요. 판별 결과를 반환 컬럼 별칭 **`connected`** 로 받으면 `rows1_5[0]['connected']` 가 답입니다.
- 그리고 아래 **서술 답안 셀**에 왜 그런 결과가 나오는지 한두 문장으로 적으세요.

**출력 형태**: `연결 여부: True` 또는 `연결 여부: False` 처럼 참·거짓 하나가 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 노드를 MATCH 로 잡고 OPTIONAL MATCH 로 shortestPath 를 구해 경로가 null 인지 본다.

세부구현:
1. 인천창고와 목포시 두 노드를 이름으로 각각 잡는다.
2. OPTIONAL MATCH 로 둘 사이 shortestPath 를 구한다(경로가 없으면 그 값이 null 이 된다).
   2-1. 화물이 실제로 갈 수 있는 방향만 따라가도록 관계를 적는다.
3. 경로가 null 이 아닌지(IS NOT NULL)를 별칭 connected 로 반환한다. 결과를 rows1_5 에 담는다.
```

</details>

In [ ]:
# 화물은 화살표 방향으로만 가므로 -[:ROUTE*]-> 로 방향을 지켜 경로를 찾는다
rows1_5 = run_cypher("""
MATCH (a:Warehouse {name:'인천창고'}), (b:City {name:'목포시'})
OPTIONAL MATCH p = shortestPath( (a)-[:ROUTE*]->(b) )
RETURN p IS NOT NULL AS connected
""")
print('연결 여부:', rows1_5[0]['connected'])

In [ ]:
# [자가채점]
assert rows1_5[0]['connected'] == False, \
    'OPTIONAL MATCH 로 감쌌는지, p IS NOT NULL 을 별칭 connected 로 줬는지, 그리고 ROUTE 의 방향을 어떻게 적었는지 다시 보세요'
print('✅ 통과!')

**서술 모범답안**: `목포시` 로 들어오는 노선은 `광주허브` 에서 오는 것뿐이고, `광주허브` 는 `부산창고` 계열에서만 연결됩니다. `인천창고` 에서 출발하는 노선은 서울허브·대전허브 쪽으로만 뻗어 `광주허브` 에 닿지 못하므로, `인천창고 → 목포시` 경로가 없습니다(연결 여부 `False`).

**해설**: 방향 있는 `ROUTE` 를 아무리 이어도 두 계열이 만나지 않으면 경로가 없습니다. `OPTIONAL MATCH` 라서 경로가 없어도 행이 남고 `p` 가 `null` 이 되어, `p IS NOT NULL` 이 `False` 가 됩니다.

⚠️ **채점 한계(강사 확인용)**: 이 채점은 `connected` 값만 봅니다. `MATCH` 로 경로를 찾아 **파이썬에서 행 수가 0인지** 보고 `False` 를 손으로 넣어도 통과합니다(실측). 그 방법은 결과가 여러 짝일 때 곧 무너지므로, 제출 코드에 `OPTIONAL MATCH` 와 `p IS NOT NULL` 이 있는지 확인하세요.

**흔한 실수**: 화살표를 빼고 `-[:ROUTE*]-` 로 물으면 결과가 **`True` 로 뒤집힙니다**. `인천창고 → 대전허브 → 전주시` 까지 간 뒤 `광주허브 → 전주시` 노선을 **거꾸로 거슬러** 광주허브로 올라가 목포시에 닿기 때문입니다. 화물은 그렇게 갈 수 없으니, 방향 있는 관계에서는 화살표를 지켜야 질문의 뜻이 맞습니다.

## 1-6. 느린 구간이 낀 경로 가려내기: `any` 와 `none`
**배경**: 1-3 은 `all` 로 "**모든** 구간이 조건을 만족하는" 경로를 골랐습니다. 이번에는 반대쪽 두 질문입니다. "**한 구간이라도** 느린 경로"와 "**하나도** 느리지 않은 경로".

**요구사항**:
- 1-3 과 같은 패턴(`(w:Warehouse)-[:ROUTE*1..2]->(c:City)`)을 경로 변수 `p` 에 담으세요.
- **`rows1_6_any`**: `any(x IN relationships(p) WHERE x.time > 80)` 으로 **한 구간이라도 80분을 넘는** 경로만 남기세요.
- **`rows1_6_none`**: 같은 조건을 `none(...)` 으로 바꿔 **하나도 넘지 않는** 경로만 남기세요.
- 둘 다 지나는 노드 이름 목록을 반환 컬럼 별칭 **`names`** 로 받으세요.

**예시**: `any` 는 **2행**, `none` 은 **4행** 입니다.

> 세는 단위는 **경로**입니다. 같은 (창고, 도시) 짝이라도 가는 길이 둘이면 두 행으로 셉니다. 그래서 같은 짝이 `any` 쪽과 `none` 쪽에 각각 나올 수 있습니다(느린 길과 빠른 길이 둘 다 있는 경우).

<details><summary>힌트</summary>

```text
접근방법:
- 1-3 의 쿼리를 그대로 두고 조건 함수만 any / none 으로 바꾼다.

세부구현:
1. MATCH p = (w:Warehouse)-[:ROUTE*1..2]->(c:City) 로 경로를 연다.
2. WHERE any(x IN relationships(p) WHERE x.time > 80) 으로 한 번, none(...) 으로 한 번 거른다.
3. 둘 다 [n IN nodes(p) | n.name] 을 별칭 names 로 반환하고 각각 rows1_6_any·rows1_6_none 에 담는다.
```

</details>

In [ ]:
# any: 목록의 원소가 하나라도 조건에 맞으면 참. '느린 구간이 낀 경로' 를 고른다
rows1_6_any = run_cypher("""
MATCH p = (w:Warehouse)-[:ROUTE*1..2]->(c:City)
WHERE any(x IN relationships(p) WHERE x.time > 80)
RETURN [n IN nodes(p) | n.name] AS names
""")
# none: 하나도 맞지 않아야 참. any 의 정반대다(조건 문장은 글자까지 같다)
rows1_6_none = run_cypher("""
MATCH p = (w:Warehouse)-[:ROUTE*1..2]->(c:City)
WHERE none(x IN relationships(p) WHERE x.time > 80)
RETURN [n IN nodes(p) | n.name] AS names
""")
# 돌아오는 순서는 정해져 있지 않으므로 파이썬에서 정렬해 찍는다
for names in sorted(r['names'] for r in rows1_6_any):
    print('느린 구간 있음:', ' → '.join(names))
for names in sorted(r['names'] for r in rows1_6_none):
    print('전 구간 80분 이하:', ' → '.join(names))

In [ ]:
# [자가채점]
assert sorted(r['names'] for r in rows1_6_any) == [['인천창고', '대전허브', '전주시'], ['인천창고', '대전허브', '천안시']], \
    'any 로 걸렀는지, 기준을 x.time > 80 으로 줬는지, 별칭이 names 인지 확인하세요'
assert sorted(r['names'] for r in rows1_6_none) == [['부산창고', '광주허브', '목포시'], ['부산창고', '광주허브', '전주시'], ['인천창고', '서울허브', '수원시'], ['인천창고', '서울허브', '천안시']], \
    'none 으로 걸렀는지(조건 안쪽에 NOT 을 넣은 any(x IN ... WHERE NOT ...) 와 혼동하지 마세요), 별칭이 names 인지 확인하세요'
print('✅ 통과!')

**해설**: `any` 는 **하나라도** 맞으면 참, `none` 은 **하나도** 안 맞아야 참입니다. 그래서 같은 조건을 주면 두 결과가 서로 겹치지 않습니다.

`none(조건)` 은 `all(반대 조건)` 과 같습니다. 여기서는 `all(x IN relationships(p) WHERE x.time <= 80)` 으로 써도 같은 답이 나옵니다. 읽기 좋은 쪽을 고르면 됩니다.

⚠️ `WHERE NOT any(...)` 로 써도 `none(...)` 과 같은 답이 나오지만, 읽는 사람에게는 `none` 쪽이 훨씬 또렷합니다.

## 1-7. 육상이든 항공이든: 관계 종류 열어 두기
**배경**: 1-5 에서 `인천창고 → 목포시` 는 육상 노선(`ROUTE`)만으로는 **닿지 못했습니다**. 그런데 이 배송망에는 항공 노선(`AIR_ROUTE`)도 있습니다. **노선 종류를 열어 두면** 답이 달라집니다.

**요구사항**:
- `인천창고` 에서 **한 홉**으로 닿는 곳을 찾되, 관계 종류를 **`-[:ROUTE|AIR_ROUTE]->`** 로 **둘 다 허용**하세요(세로선 `|` 은 "이 관계든 저 관계든"이라는 뜻입니다).
- 그 행이 **어느 노선을 탄 것인지** `type()` 으로 함께 받으세요.
- 결과를 변수 **`rows1_7`** 에 담으세요. 관계 종류를 반환 컬럼 별칭 **`kind`**, 닿는 곳 이름을 **`name`** 으로.

**예시**: **3행**이 나옵니다. 육상만 봤다면 2곳인데, 항공을 열어 **목포시**가 하나 더해집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 대괄호 안에 종류를 세로선으로 나열하면 그중 아무거나 허용된다.

세부구현:
1. MATCH (:Warehouse {name:'인천창고'})-[x:ROUTE|AIR_ROUTE]->(n) 처럼 관계에 변수 x 를 붙여 잡는다.
2. type(x) 를 별칭 kind 로, n.name 을 별칭 name 으로 반환한다.
3. 결과를 rows1_7 에 담는다. 순서는 정해져 있지 않으니 출력할 때 파이썬에서 정렬한다.
```

</details>

In [ ]:
# 세로선(|)은 단계를 늘리는 것이 아니라 한 칸짜리 관계의 '종류' 를 열어 두는 표기다
# 관계에 변수 x 를 붙여야 type(x) 로 어느 노선을 탄 것인지 되짚을 수 있다
rows1_7 = run_cypher("""
MATCH (:Warehouse {name:'인천창고'})-[x:ROUTE|AIR_ROUTE]->(n)
RETURN type(x) AS kind, n.name AS name
""")
for r in sorted(rows1_7, key=lambda x: (x['kind'], x['name'])):
    print(r['kind'], '->', r['name'])

In [ ]:
# [자가채점]
assert sorted((r['kind'], r['name']) for r in rows1_7) == [('AIR_ROUTE', '목포시'), ('ROUTE', '대전허브'), ('ROUTE', '서울허브')], \
    '관계 종류를 세로선으로 둘 다 열었는지(-[x:ROUTE|AIR_ROUTE]->), 관계에 변수를 붙여 type() 을 썼는지, 별칭이 kind·name 인지 확인하세요'
print('✅ 통과!')

**해설**: `-[:A|B]->` 는 **한 칸**인데 종류만 둘로 열어 둔 것입니다. `-[:A]->()-[:B]->` 처럼 **이어 붙이는 것**(두 단계)과는 완전히 다릅니다.

1-5 와 나란히 보면 요점이 드러납니다. **관계 종류를 어디까지 허용하느냐가 곧 질문의 범위**입니다. "화물차로 갈 수 있나"와 "어떻게든 보낼 수 있나"는 다른 질문이고, 그 차이가 대괄호 안 한 글자로 갈립니다.

`type(x)` 는 그 행이 어떤 관계를 탄 것인지 알려 줍니다. 종류를 열어 뒀을 때는 **어느 쪽으로 잡혔는지** 함께 받아 두는 편이 안전합니다.

⚠️ **채점 한계(강사 확인용)**: 조회를 **두 번 따로 돌려 파이썬에서 합쳐도** 같은 답이 나와 통과합니다(실측). 종류가 셋 넷으로 늘거나 가변길이와 섞이면 그 방법은 금세 감당이 안 되므로, 제출 코드가 `-[:ROUTE|AIR_ROUTE]->` 한 패턴인지 확인하세요.

---
# 2. 관계와 조건으로 거르기

관계를 잇고, 관계의 유무와 여러 조건으로 거릅니다(교안_02 1~3절).

## 2-1. 부산창고에서 허브를 거쳐 닿는 도시: 한 패턴으로
**배경**: `부산창고` 에서 허브를 **한 번 거쳐** 도시로 가는 노선을, 거치는 허브와 함께 나열합니다. 이번에는 단계를 나누지 말고 **관계를 한 줄로 이어 붙여** 한 패턴으로 물어봅니다.

**요구사항**:
- `Warehouse` → `Hub` → `City` 를 **한 패턴에 이어 붙여** 쓰세요. `ROUTE` 관계를 두 번 연달아 적고, 중간 노드에는 `Hub` 라벨을, 끝 노드에는 `City` 라벨을 붙입니다.
- 출발 창고는 `부산창고` 로 고정합니다. `ROUTE` 는 방향 있는 노선이니 **화살표를 지켜** 적으세요.
- 결과를 변수 **`rows2_1`** 에 담으세요. 허브 이름을 반환 컬럼 별칭 **`hub`**, 도시 이름을 **`city`** 로.

**예시**: (허브, 도시) 짝이 **2쌍** 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 창고에서 허브로, 허브에서 도시로 가는 두 관계를 한 MATCH 줄에 연달아 이어 붙인다.

세부구현:
1. 출발 노드를 이름으로 부산창고에 고정한다.
2. ROUTE 를 화살표 방향으로 한 번 타 중간 노드를 잡고, 그 중간 노드에 Hub 라벨을 붙인다.
3. 같은 줄에서 ROUTE 를 한 번 더 타 끝 노드를 잡고 City 라벨을 붙인다.
4. 허브 이름(hub)·도시 이름(city)을 반환한다. 결과를 rows2_1 에 담는다.
```

</details>

In [ ]:
# 관계 두 종류(창고->허브, 허브->도시)를 한 패턴에 연달아 이어 붙인다
rows2_1 = run_cypher("""
MATCH (:Warehouse {name:'부산창고'})-[:ROUTE]->(h:Hub)-[:ROUTE]->(c:City)
RETURN h.name AS hub, c.name AS city
""")
print(sorted((r['hub'], r['city']) for r in rows2_1))

In [ ]:
# [자가채점]
assert sorted((r['hub'], r['city']) for r in rows2_1) == [('광주허브', '목포시'), ('광주허브', '전주시')], \
    '창고→허브→도시를 한 패턴에 이어 붙였는지, 중간·끝 라벨(Hub·City)과 화살표 방향을 지켰는지 확인하세요'
print('✅ 통과!')

**해설**: 관계 종류가 같든 다르든 **화살표를 계속 이어 붙이면** 여러 단계를 한 패턴으로 물을 수 있습니다. 중간 노드에 라벨(`:Hub`)을 붙여 "허브를 거친" 경로만 남긴 것이 핵심입니다.

**흔한 실수**: `*2..2` 로 두 홉을 한 번에 건너뛰면 **중간 노드에 이름을 붙일 수 없어** 허브 이름을 꺼내지 못합니다. 중간을 결과에 써야 할 때는 이렇게 한 칸씩 적습니다.

⚠️ **채점 한계(강사 확인용)**: 이 그래프에서는 `부산창고` 에서 나가는 노선이 광주허브 하나뿐이라, 중간 라벨(`:Hub`)을 빼거나 화살표를 빼도 **결과가 같습니다.** 결과 기반 자가채점으로는 그 두 가지를 강제하지 못합니다(두 홉을 이어 붙이는 것 자체는 강제됩니다). 제출 코드에 중간 라벨과 화살표가 있는지 함께 확인하세요.

**대안 풀이**: 다음 문제처럼 `WITH` 로 단계를 나눠도 같은 답이 나옵니다. 단계를 나눌 이유가 없다면 이렇게 한 패턴으로 쓰는 쪽이 짧습니다.

## 2-2. 배송이 닿지 않는 도시: `OPTIONAL MATCH`
**배경**: 어떤 도시는 들어오는 노선이 하나도 없어 배송이 닿지 않습니다. 그런 **고립된 도시**를 찾습니다.

**요구사항**:
- 모든 `City` 를 `c` 로 잡고, 그 도시로 **들어오는** `ROUTE` 를 `OPTIONAL MATCH` 로 이으세요. 이때 노선의 **출발 노드에 변수 `x` 를 붙입니다**(들어오는 노선이 없으면 그 `x` 가 `null` 이 됩니다). 그다음 **`WITH c, x` 로 한 단계 넘긴 뒤** `x` 가 `null` 인 도시만 남기세요(교안_02 3-2. `OPTIONAL MATCH` 에 `WHERE` 를 바로 붙이면 걸러지지 않습니다).
- 결과를 변수 **`rows2_2`** 에 담으세요. 도시 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: 고립된 도시는 **['제주시']** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모든 City 를 남긴 뒤 OPTIONAL MATCH 로 인바운드 노선을 잇고, 그 짝이 null 인 것만 거른다.

세부구현:
1. 먼저 모든 City 를 잡아 둔다(이 도시들은 반드시 남아야 한다).
2. OPTIONAL MATCH 로 그 도시로 들어오는 ROUTE 의 출발 노드를 (있으면) 잇고, 그 출발 노드에 변수 x 를 붙인다.
3. WITH c, x 로 한 단계 넘긴 다음 줄에서 x 가 null 인 도시만 거르고, 이름을 별칭 name 으로 반환한다. 결과를 rows2_2 에 담는다.
```

</details>

In [ ]:
# 모든 도시를 남긴 뒤 인바운드 노선이 null 인 도시만 거른다
rows2_2 = run_cypher("""
MATCH (c:City)
OPTIONAL MATCH (x)-[:ROUTE]->(c)
WITH c, x WHERE x IS NULL
RETURN c.name AS name
""")
print(sorted(r['name'] for r in rows2_2))

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_2) == ['제주시'], \
    'OPTIONAL MATCH 로 도시를 먼저 남겼는지, WITH 로 넘긴 뒤 WHERE x IS NULL 로 걸렀는지 확인하세요 ' \
    '(WHERE 를 OPTIONAL MATCH 에 바로 붙이면 모든 도시가 나옵니다)'
print('✅ 통과!')

⚠️ **채점 한계(강사 확인용)**: `WHERE NOT (:Warehouse|Hub|City)-[:ROUTE]->(c)` 처럼 **패턴 술어**로 써도 같은 답이 나와 통과합니다(실측). 다만 그 방법은 "짝이 없는 행을 남긴다" 는 `OPTIONAL MATCH` 의 성질을 안 쓰고, 3-1 처럼 **짝을 결과에 함께 담아야 할 때** 쓸 수 없습니다. 제출 코드에 `OPTIONAL MATCH` 와 `WITH` 가 있는지 확인하세요.

**해설**: 보통 `MATCH` 로 인바운드 노선을 이으면 노선 없는 도시는 사라집니다. `OPTIONAL MATCH` 로 도시를 남긴 뒤 `x IS NULL` 로 걸러야 고립 도시를 찾을 수 있습니다. 이때 `WHERE` 를 `OPTIONAL MATCH` 줄에 바로 붙이면 그 `WHERE` 가 패턴을 찾는 조건이 되어 아무것도 걸러지지 않고 **모든 도시**가 나옵니다. `WITH c, x` 로 한 단계 넘긴 뒤에 걸어야 합니다.

## 2-3. 천안행 노선이 없는 허브: `EXISTS { }`
**배경**: 천안 물량을 늘리려고 합니다. **천안시로 가는 노선이 하나도 없는 허브**를 찾아 신설 후보로 삼습니다.

**요구사항**:
- 허브를 잡고, `WHERE` 에 **`NOT EXISTS { (h)-[:ROUTE]->(:City {name:'천안시'}) }`** 를 걸어 천안행 노선이 없는 허브만 남기세요.
- 결과를 변수 **`rows2_3`** 에 담으세요. 허브 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **1곳**(광주허브)이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계가 '없는' 것은 MATCH 로 이어서는 못 찾는다. 조건 자리에서 패턴의 존재를 묻고 NOT 으로 뒤집는다.

세부구현:
1. MATCH (h:Hub) 로 허브를 모두 잡는다.
2. WHERE NOT EXISTS { (h)-[:ROUTE]->(:City {name:'천안시'}) } 를 적는다.
3. h.name 을 별칭 name 으로 반환한다. 결과를 rows2_3 에 담는다.
```

</details>

In [ ]:
# 중괄호 안의 패턴이 하나라도 맞으면 EXISTS 는 참이다. NOT 을 붙여 '하나도 없는' 허브만 남긴다
rows2_3 = run_cypher("""
MATCH (h:Hub)
WHERE NOT EXISTS { (h)-[:ROUTE]->(:City {name:'천안시'}) }
RETURN h.name AS name
""")
print(sorted(r['name'] for r in rows2_3))

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_3) == ['광주허브'], \
    'EXISTS 중괄호 안에 패턴을 적고 앞에 NOT 을 붙였는지, 화살표 방향이 허브->도시 인지, 별칭이 name 인지 확인하세요'
print('✅ 통과!')

**해설**: 이 문항은 패턴 술어(`WHERE NOT (h)-[:ROUTE]->(:City {name:'천안시'})`)로도 똑같이 풀립니다. 도시 쪽 조건이 이름 하나로 딱 정해져 있어 속성 map 으로 적을 수 있기 때문입니다. `EXISTS { }` 가 **꼭 필요한** 때는 상대 쪽에 `STARTS WITH` 같은 조건을 걸어야 해서 중괄호 안에서 변수를 만들어야 할 때입니다.

## 2-4. 허브에서 출발하는 느린 노선: 복합 필터
**배경**: 점검 대상을 좁힙니다. **허브에서 출발하면서** 동시에 **오래 걸리는** 노선만 보려고 합니다. 조건이 둘이니 한 `WHERE` 에 묶어야 합니다.

**요구사항**:
- 모든 `ROUTE` 중에서, **출발지 이름이 `'허브'` 로 끝나고**(`ENDS WITH`) **동시에** `time` 이 **55 이상**인 노선만 남기세요. 두 조건을 한 `WHERE` 에 `AND` 로 묶습니다.
- 결과를 변수 **`rows2_4`** 에 담으세요. 출발 이름을 반환 컬럼 별칭 **`a`**, 도착 이름을 **`b`**, 시간을 **`t`** 로, **`time` 내림차순, 동점이면 출발지 이름·도착지 이름 오름차순** 정렬로.

**예시**: 노선 **3개**가 시간 내림차순으로 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모든 노선을 잡고, 출발지 이름의 끝을 보는 조건과 시간 조건을 한 WHERE 에 함께 건다.

세부구현:
1. 노선 관계와 그 양 끝 노드를 잡는다(관계에도 변수를 붙여 time 을 쓸 수 있게 한다).
2. WHERE 에 두 조건을 AND 로 묶는다.
   2-1. 출발 노드 이름이 '허브' 로 끝나는지(접미 일치 연산자)
   2-2. 관계의 time 이 55 이상인지
3. 출발 이름(a)·도착 이름(b)·시간(t)을 반환하고 ORDER BY 로 시간 내림차순, 보조키로 출발·도착 이름을 준다.
4. 결과를 rows2_4 에 담는다.
```

</details>

In [ ]:
# 접미 일치와 숫자 조건을 한 WHERE 에 AND 로 묶는다
rows2_4 = run_cypher("""
MATCH (a)-[r:ROUTE]->(b)
WHERE a.name ENDS WITH '허브' AND r.time >= 55
RETURN a.name AS a, b.name AS b, r.time AS t
ORDER BY r.time DESC, a.name, b.name
""")
print([(r['a'], r['b'], r['t']) for r in rows2_4])

In [ ]:
# [자가채점]
assert [(r['a'], r['b'], r['t']) for r in rows2_4] == [('서울허브', '천안시', 80), ('대전허브', '전주시', 70), ('광주허브', '목포시', 55)], \
    '두 조건을 AND 로 묶었는지, ENDS WITH 로 출발지 이름의 끝을 봤는지, 정렬 기준을 다 줬는지 확인하세요'
print('✅ 통과!')

**해설**: 새로 배운 문자열 연산자도 결국 `WHERE` 안의 조건 하나입니다. 숫자 비교와 똑같이 `AND`·`OR` 로 묶습니다. 조건이 셋 이상이면 괄호로 우선순위를 못 박는 게 안전합니다.

**흔한 실수**: `AND` 대신 `OR` 를 쓰면 허브가 아닌 출발지(창고)의 느린 노선까지 들어옵니다. `WHERE` 를 두 줄로 나눠 쓰는 것도 문법 오류입니다. 한 `WHERE` 에 이어 붙이세요.

⚠️ **채점 한계(강사 확인용)**: 이 데이터에는 이름 가운데에 '허브'가 든 노드가 없어 `CONTAINS '허브'` 로 써도 결과가 같습니다. `ENDS WITH` 를 골랐는지는 제출 코드로 확인하세요. "이름이 무엇으로 끝나는가"를 묻는 질문에는 `ENDS WITH` 가 뜻이 맞습니다.

**대안 풀이**: `WHERE r.time >= 55 AND a.name ENDS WITH '허브'` 처럼 순서를 바꿔도 결과는 같습니다.

## 2-5. 빠른 노선이 하나도 없는 허브: `EXISTS { }` 안에 조건 걸기
**배경**: 급송 물량은 **30분 안에 닿는 도시**가 있어야 받을 수 있습니다. 2-3 은 "천안행 노선이 있는가" 처럼 **패턴만** 물었지만, 이번에는 노선의 `time` 까지 봐야 합니다.

**요구사항**:
- `MATCH (h:Hub)` 로 허브를 잡고, `WHERE NOT EXISTS { ... }` 로 **30분 이하 노선이 하나도 없는** 허브만 남기세요.
- 중괄호 안 패턴의 **노선에 변수를 붙여**(`-[r:ROUTE]->`) 그 뒤에 `WHERE r.time <= 30` 을 함께 적습니다.
- 결과를 변수 **`rows2_5`** 에 담으세요. 허브 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **2곳**(광주허브·대전허브)이 나옵니다.

**주의**: 교안_02 2-2 의 짧은 패턴 술어(`WHERE (h)-[:ROUTE]->(:City)`)에는 `WHERE` 를 붙일 수 없고, 속성 map 은 `{time: 30}` 처럼 **같은 값**만 적을 수 있습니다. 그래서 이 문제는 `EXISTS { }` 여야 풀립니다.

<details><summary>힌트</summary>

```text
접근방법:
- '하나도 없다' 는 조건 자리에서 존재를 묻고 뒤집는다. 그런데 이번엔 존재만이 아니라 노선의 시간까지 본다.

세부구현:
1. MATCH (h:Hub) 로 허브를 모두 잡는다.
2. WHERE NOT EXISTS { ... } 를 적고, 중괄호 안에 (h)-[r:ROUTE]->(:City) 패턴을 넣는다.
3. 같은 중괄호 안, 패턴 뒤에 WHERE r.time <= 30 을 이어 적는다.
4. h.name 을 별칭 name 으로 반환한다. 결과를 rows2_5 에 담는다.
```

</details>

In [ ]:
# 중괄호 안에서는 변수를 만들 수 있고 WHERE 도 쓸 수 있다. 여기서는 노선에 r 을 붙여 시간을 본다
# NOT 을 앞에 붙였으니 '30분 이하 노선이 하나라도 있는' 허브가 빠지고 나머지만 남는다
rows2_5 = run_cypher("""
MATCH (h:Hub)
WHERE NOT EXISTS { (h)-[r:ROUTE]->(:City) WHERE r.time <= 30 }
RETURN h.name AS name
""")
print(sorted(r['name'] for r in rows2_5))

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_5) == ['광주허브', '대전허브'], \
    '중괄호 안에 노선 변수를 붙이고 그 뒤에 WHERE r.time <= 30 을 적었는지, 앞에 NOT 을 붙였는지, 별칭이 name 인지 확인하세요'
print('✅ 통과!')

**해설**: 2-3 은 `EXISTS { }` 를 썼지만 사실 **교안_02 2-2 의 짧은 표기로도** 쓸 수 있는 모양이었습니다(`WHERE NOT (h)-[:ROUTE]->(:City {name:'천안시'})`). 이 문제는 다릅니다. 노선의 `time` 을 **비교**해야 하는데, 짧은 표기에는 `WHERE` 를 붙일 자리가 없고 속성 map 은 같은 값만 적을 수 있습니다. **중괄호 안에서 변수를 만들고 조건을 거는 것**이 `EXISTS { }` 의 존재 이유입니다.

**답이 갈리는 지점**: 서울허브는 수원시로 가는 30분 노선이 있어 빠지고, 광주허브·대전허브만 남습니다.

**흔한 실수**: `WHERE` 를 중괄호 **밖**에 적으면(`NOT EXISTS { ... } WHERE r.time <= 30`) `r` 이 중괄호 밖으로 나오지 못해 오류가 납니다. 조건은 중괄호 **안**, 패턴 바로 뒤입니다.

**대안 풀이**: `MATCH (h:Hub)-[r:ROUTE]->(:City) WHERE r.time <= 30` 으로 **빠른 노선이 있는** 허브를 먼저 구한 뒤 손으로 빼는 방법도 있지만, "하나도 없다" 를 한 조회로 표현하려면 `NOT EXISTS { }` 가 필요합니다.

---
# 3. `WITH` 파이프라인

중간 결과를 넘기고, 값을 새로 만들고, 잘라서 잇습니다(교안_02 4절).

## 3-1. 가장 빠른 허브 한 곳만 골라 도시 잇기: `WITH` 체인
**배경**: 2-1 은 한 패턴으로 끝났습니다. 이번에는 **중간에서 한 번 추려야** 하는 질문입니다. `인천창고` 에서 나가는 노선 중 **가장 빠른 노선의 허브 한 곳만** 고르고, **그 허브가 닿는 도시를 모두** 나열하세요.

**요구사항**:
- 1단계: `인천창고` 에서 한 홉으로 닿는 `Hub` 를 노선의 `time` 이 **작은 순**으로 줄 세워 **한 곳만** 남기고, 그것을 `WITH` 로 다음 단계에 넘기세요. **무엇으로 줄 세웠는지가 `WITH` 줄에 남도록** 정렬 기준도 함께 넘기고 `AS` 로 이름을 붙이세요.
- 2단계: 넘긴 허브에서 한 홉으로 닿는 `City` 를 이으세요.
- 결과를 변수 **`rows3_1`** 에 담으세요. 허브 이름을 반환 컬럼 별칭 **`hub`**, 도시 이름을 **`city`** 로.

**예시**: 허브 한 곳에 도시 **2곳**, 즉 (허브, 도시) 짝 **2쌍**이 나옵니다.

**주의**: 정렬과 "한 곳만"이 **중간 단계**에 걸립니다. 마지막 `RETURN` 에 `LIMIT 1` 을 붙이면 도시까지 한 줄로 잘려 **1행**만 남습니다. 그래서 이 문제는 `WITH` 없이는 풀 수 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 창고에서 허브로 가는 노선을 찾아 시간순으로 줄 세우고 한 곳만 남긴 뒤, 그 허브를 다음 단계로 넘긴다.

세부구현:
1. 인천창고에서 한 홉으로 닿는 Hub 를 찾되, 그 노선 관계에도 변수를 붙여 시간을 쓸 수 있게 한다.
2. WITH 로 허브와 노선 시간을 함께 넘기면서, 그 자리에서 시간 오름차순으로 정렬하고 하나만 남긴다.
   2-0. 노선 시간에는 AS 로 이름을 붙여 넘기고, ORDER BY 는 그 이름으로 건다.
   2-1. 넘길 때 정렬·개수 제한을 함께 적을 수 있다(WITH 뒤에 ORDER BY, LIMIT 순).
3. 넘긴 허브에서 한 홉으로 닿는 City 를 잇고, 허브 이름(hub)·도시 이름(city)을 반환한다. 결과를 rows3_1 에 담는다.
```

</details>

In [ ]:
# 중간 단계에서 정렬해 허브 하나만 남기고(WITH ... ORDER BY ... LIMIT) 그 뒤에 도시를 잇는다
# 정렬 기준인 노선 시간도 WITH 에 함께 적는다. 적지 않으면 그 뒤에서 r 은 사라진다
rows3_1 = run_cypher("""
MATCH (:Warehouse {name:'인천창고'})-[r:ROUTE]->(h:Hub)
WITH h, r.time AS t ORDER BY t LIMIT 1
MATCH (h)-[:ROUTE]->(c:City)
RETURN h.name AS hub, c.name AS city
""")
print(sorted((r['hub'], r['city']) for r in rows3_1))

In [ ]:
# [자가채점]
assert sorted((r['hub'], r['city']) for r in rows3_1) == [('서울허브', '수원시'), ('서울허브', '천안시')], \
    'WITH 로 넘기면서 그 자리에서 시간순 정렬하고 하나만 남겼는지 확인하세요. 마지막 RETURN 에 LIMIT 을 붙이면 답이 달라집니다'
print('✅ 통과!')

**해설**: `WITH h, r.time AS t ORDER BY t LIMIT 1` 이 이 문제의 전부입니다. `WITH` 는 값을 넘기기만 하는 게 아니라 **넘기면서 정렬하고 개수를 자를** 수 있습니다. 그렇게 추린 허브 하나를 다음 `MATCH` 가 이어받아 도시로 뻗습니다.

**정렬 기준도 함께 넘깁니다.** `WITH` 에 적은 것만 그 줄 **다음**으로 넘어가므로, 뒤에서 `r.time` 을 쓰려면 `r.time AS t` 처럼 이름을 붙여 넘겨야 합니다(교안_02 4-2 의 `AS`). 아래 대안 풀이도 함께 읽어 보세요.

**흔한 실수**: 2-1 처럼 한 패턴으로 쓰고 마지막에 `ORDER BY r.time LIMIT 1` 을 붙이면 **1행**만 남습니다. 자르는 시점이 다르기 때문입니다. 허브를 먼저 하나로 줄이고 나서 도시를 뻗어야 합니다.

**대안 풀이**: `WITH h ORDER BY r.time LIMIT 1` 처럼 시간을 안 넘겨도 **돌아갑니다.** `ORDER BY` 는 `WITH` 절의 일부라 아직 **넘기기 전** 장면을 볼 수 있어서 `r` 이 살아 있습니다. 다만 그 줄을 지나면 `r` 은 사라지고, 읽는 사람은 무엇으로 줄 세웠는지 되짚어 올라가야 합니다. 그래서 이 풀이는 정렬 기준을 함께 넘깁니다.

⚠️ **채점 한계(강사 확인용)**: 이 채점은 결과 짝만 봅니다. 데이터 살펴보기 화면에서 인천창고발 최저 시간이 서울허브라는 것을 눈으로 읽고 허브 이름을 그대로 적어 넣으면, `WITH` 도 정렬도 쓰지 않고 통과합니다. 제출한 답안에 `WITH ... ORDER BY ... LIMIT` 이 실제로 있는지 확인하세요.

2-1(한 패턴)과 3-1(`WITH` 체인)을 나란히 보면, **단계를 나눌 이유가 있을 때만** `WITH` 를 쓴다는 기준이 잡힙니다.

## 3-2. 천안행 노선 중 빠른 것만: 2단계 필터
**배경**: `천안시` 로 들어오는 노선 중 **60분 이하**로 빠른 노선만 골라, 어디서 출발하는지 봅니다.

**요구사항**:
- `천안시` 로 들어오는 `ROUTE` 를 찾고, `WITH` 로 **출발지 이름과 소요 시간**을 넘긴 뒤 **`time <= 60`** 인 것만 남기세요.
- 결과를 변수 **`rows3_2`** 에 담으세요. 출발지 이름을 반환 컬럼 별칭 **`src`**, 시간을 **`t`** 로, 소요 시간 오름차순 정렬로.

**예시**: 60분 이하인 (출발지, 시간) 쌍 **1개**가 시간 오름차순으로 나옵니다(80분짜리 노선은 걸러집니다).

<details><summary>힌트</summary>

```text
접근방법:
- 천안행 노선을 찾고 WITH 로 출발지·시간을 넘긴 뒤 그 단계에서 WHERE 로 시간을 거른다.

세부구현:
1. 도착이 천안시인 ROUTE 와 그 출발 노드를 잡는다.
2. WITH 로 출발지 이름(src)과 관계의 time(t) 만 추려 넘기면서, 그 자리에서 t 가 60 이하인 것만 WHERE 로 남긴다.
3. src·t 를 반환하고 t 오름차순 정렬한다. 결과를 rows3_2 에 담는다.
```

</details>

In [ ]:
# WITH 로 출발지·시간만 추려 넘기면서 그 자리에서 시간을 한 번 더 거른다
rows3_2 = run_cypher("""
MATCH (x)-[r:ROUTE]->(:City {name:'천안시'})
WITH x.name AS src, r.time AS t WHERE t <= 60
RETURN src, t
ORDER BY t
""")
print([(r['src'], r['t']) for r in rows3_2])

In [ ]:
# [자가채점]
assert [(r['src'], r['t']) for r in rows3_2] == [('대전허브', 50)], \
    'WITH 로 출발지 이름과 시간만 넘겼는지, 그 단계에서 60 이하로 걸렀는지, 시간 오름차순인지 확인하세요'
print('✅ 통과!')

**해설**: `WITH` 로 필요한 값(출발지·시간)만 추려 넘기고 거기에 `WHERE` 를 겁니다. `MATCH` 의 `WHERE` 와 달리 **중간 단계에서 한 번 더** 거를 수 있는 게 `WITH` 의 힘입니다.

⚠️ **채점 한계(강사 확인용)**: 이 문항은 조건이 둘뿐이라 `WHERE ... AND ...` 한 줄로 써도 같은 답이 나와 통과합니다(실측). `WITH` 가 **없으면 못 푸는** 자리는 3-1(중간에서 잘라 다음 단계를 잇기)입니다. 여기서는 단계를 나누는 연습으로 보고, 제출 코드에 `WITH` 가 있는지 확인하세요.

## 3-3. 두 구간 합계로 거르기: `WITH` 로 값 만들기
**배경**: 1-4 에서 같은 길이의 경로가 둘이었습니다. 무엇을 고를지 정하려면 **총 소요 시간**이라는 기준이 필요한데, 그 값은 데이터에 없습니다. 두 구간을 **더해서** 만들어야 합니다.

**요구사항**:
- `(w:Warehouse)-[r1:ROUTE]->(h:Hub)-[r2:ROUTE]->(c:City)` 로 두 구간을 잇고, **`WITH w, c, r1.time + r2.time AS 총시간`** 으로 합계를 만들어 넘기세요.
- 그다음 줄 `WHERE 총시간 <= 130` 으로 거르세요.
- 결과를 변수 **`rows3_3`** 에 담으세요. 창고 이름 **`w`**, 도시 이름 **`c`**, 합계 **`t`** 별칭으로 반환하고 **`t` 오름차순, 같으면 창고·도시 이름 오름차순**으로 정렬하세요.

**예시**: **4행**이 나오고 맨 위는 **인천창고 → 수원시 (70분)** 입니다.

> `WITH` 의 표현식에는 **`AS` 로 이름을 붙여야** 합니다. 그리고 `WITH` 에 적지 않은 변수는 다음 절에서 사라지므로 `w`·`c` 도 함께 넘겨야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 데이터에 없는 값(합계)을 WITH 에서 만들고, 그 이름으로 다음 줄에서 거른다.

세부구현:
1. 창고->허브->도시 두 구간을 관계 변수 r1·r2 로 잡는다.
2. WITH w, c, r1.time + r2.time AS 총시간 으로 넘긴다(w·c 를 빼먹으면 뒤에서 못 쓴다).
3. WHERE 총시간 <= 130 으로 거르고, w·c·총시간 을 별칭 w·c·t 로 반환한다.
4. ORDER BY t, w, c 로 정렬해 rows3_3 에 담는다.
```

</details>

In [ ]:
# 총 소요 시간은 데이터에 없는 값이라 WITH 에서 더해 이름을 붙인다
# w·c 를 함께 넘겨야 마지막 RETURN 에서 쓸 수 있다
rows3_3 = run_cypher("""
MATCH (w:Warehouse)-[r1:ROUTE]->(h:Hub)-[r2:ROUTE]->(c:City)
WITH w, c, r1.time + r2.time AS 총시간
WHERE 총시간 <= 130
RETURN w.name AS w, c.name AS c, 총시간 AS t
ORDER BY t, w, c
""")
for r in rows3_3:
    print(r['w'], '→', r['c'], f"({r['t']}분)")

In [ ]:
# [자가채점]
assert [(r['w'], r['c'], r['t']) for r in rows3_3] == [('인천창고', '수원시', 70), ('부산창고', '전주시', 110), ('인천창고', '천안시', 120), ('부산창고', '목포시', 125)], \
    'WITH 에서 두 구간을 더해 AS 로 이름을 붙였는지, w·c 도 함께 넘겼는지, 별칭이 w·c·t 인지, 정렬을 t·w·c 로 줬는지 확인하세요'
print('✅ 통과!')

**해설**: `총시간` 은 데이터에 없는 열입니다. `WITH` 에서 **한 번 만들어 이름을 붙였기** 때문에 그다음 줄의 `WHERE`·`ORDER BY`·`RETURN` 에서 모두 그 이름으로 부를 수 있었습니다.

⚠️ **채점 한계(강사 확인용)**: `WITH` 없이 `WHERE r1.time + r2.time <= 130` 으로 써도 **같은 답이 나옵니다**(실측). `WITH` 가 여기서 주는 것은 "못 하던 일" 이 아니라 **같은 식을 세 번 되풀이하지 않는 것**입니다. 되풀이해 쓴 답안도 정답으로 인정하되, 이름을 붙이는 쪽이 왜 나은지는 짚어 주세요. "`AND` 로는 못 한다" 가 진짜인 자리는 3-1(중간에서 잘라 다음 단계를 잇기)입니다.

1-4 에서 두 갈래였던 `인천창고 → 천안시` 는 서울허브를 거치면 120분, 대전허브를 거치면 140분입니다. **같은 단계 수라도 걸리는 시간은 다릅니다.** 이 표에 남은 것은 130분 조건을 통과한 120분짜리 한 줄뿐이고, 140분짜리는 걸러졌습니다.

---
# 4. 정렬과 쪽 넘기기

정렬해 상위를 뽑고 그다음 쪽으로 넘어갑니다(교안_02 5절).

## 4-1. 가장 오래 걸리는 노선 세 개: 정렬 상위 N
**배경**: 소요 시간이 긴 노선 상위 세 개를 뽑아 점검 대상으로 삼습니다.

**요구사항**:
- 모든 `ROUTE` 를 **`time` 내림차순**으로, **동점이면 출발지 이름·도착지 이름 오름차순**으로 정렬해 **상위 3개**(`LIMIT 3`)를 뽑으세요.
- 결과를 변수 **`rows4_1`** 에 담으세요. 출발 이름을 반환 컬럼 별칭 **`a`**, 도착 이름을 **`b`**, 시간을 **`t`** 로.

**예시**: (출발, 도착, 시간) 세 값이 담긴 노선 **3개**가 시간 내림차순으로 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모든 ROUTE 를 시간 내림차순으로 정렬하고 보조키(출발·도착 이름)를 더한 뒤 LIMIT 로 자른다.

세부구현:
1. 모든 ROUTE 와 그 양 끝 노드를 잡아 출발 이름(a)·도착 이름(b)·time(t) 을 반환한다.
2. ORDER BY 에 time 내림차순(DESC), 보조키로 출발 이름·도착 이름 오름차순을 함께 준다.
3. LIMIT 3 으로 상위 셋만 남긴다. 결과를 rows4_1 에 담는다.
```

</details>

In [ ]:
# 시간 내림차순 + 이름 오름차순(보조 정렬키)으로 정렬해 상위 3개만 남긴다
rows4_1 = run_cypher("""
MATCH (a)-[r:ROUTE]->(b)
RETURN a.name AS a, b.name AS b, r.time AS t
ORDER BY r.time DESC, a.name, b.name
LIMIT 3
""")
print([(r['a'], r['b'], r['t']) for r in rows4_1])

In [ ]:
# [자가채점]
assert [(r['a'], r['b'], r['t']) for r in rows4_1] == [('인천창고', '대전허브', 90), ('서울허브', '천안시', 80), ('대전허브', '전주시', 70)], \
    'ORDER BY 에 time 내림차순과 보조 정렬키(출발·도착 이름)를 둘 다 줬는지, LIMIT 3 인지 확인하세요'
print('✅ 통과!')

**해설**: `ORDER BY ... DESC` 로 내림차순, 보조 정렬키로 이름을 더해 동점 순서를 고정합니다. `LIMIT 3` 이 상위 세 개만 남깁니다.

## 4-2. 그다음 세 노선: `SKIP` 으로 쪽 넘기기
**배경**: 4-1 에서 가장 오래 걸리는 노선 세 개를 봤습니다. 점검 목록을 세 개씩 끊어 보여 주려고 **그다음 세 노선**(4~6위)을 뽑습니다.

**요구사항**:
- 정렬 기준은 **4-1 과 똑같이**: `time` 내림차순, **동점이면 출발지 이름·도착지 이름 오름차순**.
- 앞의 **3개를 건너뛰고**(`SKIP 3`) **3개만**(`LIMIT 3`) 뽑으세요.
- 결과를 변수 **`rows4_2`** 에 담으세요. 반환 컬럼 별칭은 4-1 과 같이 출발 **`a`**, 도착 **`b`**, 시간 **`t`** 로.

**주의**: 이 데이터에는 **70분짜리 노선이 두 개** 있고 그 둘이 하필 3·4위, 즉 **페이지 경계**에 놓입니다. 보조 정렬키가 없으면 둘 중 어느 쪽이 3위인지 **정해지지 않아** 같은 노선이 두 페이지에 겹쳐 나오거나 어느 페이지에도 안 나올 수 있습니다. 두 페이지의 `ORDER BY` 를 완전히 같게, 그리고 동점이 남지 않게 주세요.

**예시**: 4~6위 노선 **3개**가 시간 내림차순으로 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4-1 과 똑같이 정렬한 뒤, 앞의 세 개를 건너뛰고 세 개만 남긴다.

세부구현:
1. 모든 ROUTE 와 그 양 끝 노드를 잡아 출발 이름(a)·도착 이름(b)·time(t) 을 반환한다.
2. ORDER BY 를 4-1 과 완전히 똑같이 준다(time 내림차순, 보조키로 출발·도착 이름).
3. SKIP 으로 앞 세 개를 건너뛰고 LIMIT 로 세 개만 남긴다(적는 순서는 ORDER BY, SKIP, LIMIT).
4. 결과를 rows4_2 에 담는다.
```

</details>

In [ ]:
# 4-1 과 같은 정렬로 앞 3개를 건너뛰고(SKIP) 3개만 남긴다(LIMIT) = 2페이지
rows4_2 = run_cypher("""
MATCH (a)-[r:ROUTE]->(b)
RETURN a.name AS a, b.name AS b, r.time AS t
ORDER BY r.time DESC, a.name, b.name
SKIP 3 LIMIT 3
""")
print([(r['a'], r['b'], r['t']) for r in rows4_2])

In [ ]:
# [자가채점]
assert [(r['a'], r['b'], r['t']) for r in rows4_2] == [('부산창고', '광주허브', 70), ('광주허브', '목포시', 55), ('대전허브', '천안시', 50)], \
    'ORDER BY 를 4-1 과 완전히 똑같이(보조 정렬키 포함) 줬는지, SKIP 3 LIMIT 3 인지 확인하세요'
print('✅ 통과!')

**해설**: `SKIP 3 LIMIT 3` 이 "4~6번째"가 됩니다. 4-1 과 `ORDER BY` 가 완전히 같아야 두 페이지가 이어집니다. 정렬을 조금이라도 다르게 주면 겹치거나 빠지는 노선이 생깁니다.

**흔한 실수(강사 확인용)**: 보조 정렬키 없이 `ORDER BY r.time DESC` 만 준 답안이 있습니다. 이 데이터에는 70분 동점이 3·4위에 있어 그 답안의 결과가 **정해져 있지 않습니다.** Cypher 는 동점 사이 순서를 약속하지 않으므로 그런 답안은 **통과할 수도, 실패할 수도** 있습니다. 자가채점이 통과했다고 옳은 풀이는 아니니, 제출 코드에 보조 정렬키가 있는지 눈으로 확인하세요. 교안_02 5-1 이 쪽을 넘길 때 보조 정렬키가 왜 필수인지 다룹니다.

---
수고했어요! LV2 에서 경로 탐색(`all`·`allShortestPaths` 포함), 관계와 조건으로 거르기(`OPTIONAL MATCH`·`EXISTS { }`), `WITH` 파이프라인(파생값 포함), 정렬과 쪽 넘기기를 **조합**했습니다. LV3 에서는 이것들을 묶어 SNS 친구 네트워크를 분석하고 경로를 진단합니다.